# Yield regressions and VIF

Tables 3.4 and 3.6; Appendix Tables 5.3 and 5.4. Curve uses recovered source cells. The lending benchmark is a documented configuration recovery verified against the submitted table.

In [1]:
from pathlib import Path
import os

# Locate the bundle from either its root or the notebooks directory.
_here = Path.cwd().resolve()
_ROOT = next((p for p in [_here, *_here.parents]
              if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if _ROOT is None:
    raise FileNotFoundError("Open this notebook inside the Tesi_codice_consegna folder.")
_DATA_DIR = _ROOT / "data"
_OUTPUT_DIR = _ROOT / "results" / "rerun" / "01_yield_regressions"
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)
print("Data:", _DATA_DIR)
print("New outputs:", _OUTPUT_DIR)


Data: /Users/macbook/Desktop/Tesi Master/Tesi_codice_consegna/data
New outputs: /Users/macbook/Desktop/Tesi Master/Tesi_codice_consegna/results/rerun/01_yield_regressions


## Curve 3pool: parsimonious and full models

Original calculations retained. CSV paths now point to the bundled data.

Source: `AAVE v2/REGRESSIONDAUSARE.ipynb`, original cell 5 (zero-based).

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1. CONFIGURAZIONE E CARICAMENTO DATI ---

main_data_filename = str(_DATA_DIR / 'aavegas.csv')
btc_data_filename =  str(_DATA_DIR / 'btc_filtered.csv')
date_col = 'date'
dependent_var = '3pool'

# Definiamo le variabili per i due modelli
vars_parsimonious = ['3m ir']    
vars_full = ['3m ir', 'ethperpetual', 'gas_fees', 'VIX', 'btc_return', 'volatility_btc']# ---------------------------------------------------

try:
    # Carica il DataFrame principale
    df_main = pd.read_csv(main_data_filename, parse_dates=[date_col], dayfirst=True)
    df_main.set_index(date_col, inplace=True)
    
    # Carica il DataFrame con i dati di Bitcoin
    df_btc = pd.read_csv(btc_data_filename, parse_dates=["Date"], dayfirst=True)
    df_btc.set_index("Date", inplace=True)
    
    print("Dati principali e dati BTC caricati con successo.")

    df = df_main.join(df_btc, how='inner')
    df.sort_index(inplace=True)
    
    print("DataFrame uniti con successo.")

except Exception as e:
    print(f"ERRORE durante il caricamento o la fusione dei dati: {e}")
    exit()

# Rimuovi righe con dati mancanti per il modello completo (per avere lo stesso campione)
df.dropna(subset=[dependent_var] + vars_full, inplace=True)
print(f"Analisi condotta su {len(df)} osservazioni giornaliere.")

# --- 2. PREPARAZIONE DELLE VARIABILI ---

# Variabile dipendente (Y)
y = df[dependent_var]

# Variabili indipendenti (X) per entrambi i modelli
X_parsimonious = sm.add_constant(df[vars_parsimonious])
X_full = sm.add_constant(df[vars_full])


# --- 3. ESECUZIONE DELLE REGRESSIONI ---

print("\nEsecuzione del Modello Parsimonioso (solo risk-free)...")
model_parsimonious = sm.OLS(y, X_parsimonious)
results_parsimonious = model_parsimonious.fit(cov_type='HAC', cov_kwds={'maxlags': 7})

print("\nEsecuzione del Modello Completo (con tutti i controlli)...")
model_full = sm.OLS(y, X_full)
results_full = model_full.fit(cov_type='HAC', cov_kwds={'maxlags': 7})


# --- 4. VISUALIZZAZIONE COMPARATIVA DEI RISULTATI ---

print("\n" + "="*25 + " MODELLO 1: PARSIMONIOSO " + "="*25)
print(results_parsimonious.summary())
print("="*75)

print("\n" + "="*25 + " MODELLO 2: COMPLETO " + "="*27)
print(results_full.summary())
print("="*75)

# --- CREAZIONE DI UNA TABELLA DI CONFRONTO ---

# Estrai i risultati chiave in un formato pulito
comparison_data = {
    'Variable': [],
    'Model_Parsimonious_Coeff': [],
    'Model_Parsimonious_PValue': [],
    'Model_Full_Coeff': [],
    'Model_Full_PValue': [],
}

# Aggiungi i coefficienti
for var in vars_parsimonious:
    comparison_data['Variable'].append(var)
    comparison_data['Model_Parsimonious_Coeff'].append(results_parsimonious.params[var])
    comparison_data['Model_Parsimonious_PValue'].append(results_parsimonious.pvalues[var])
    comparison_data['Model_Full_Coeff'].append(results_full.params.get(var, ''))
    comparison_data['Model_Full_PValue'].append(results_full.pvalues.get(var, ''))

for var in vars_full:
    if var not in vars_parsimonious:
        comparison_data['Variable'].append(var)
        comparison_data['Model_Parsimonious_Coeff'].append('')
        comparison_data['Model_Parsimonious_PValue'].append('')
        comparison_data['Model_Full_Coeff'].append(results_full.params[var])
        comparison_data['Model_Full_PValue'].append(results_full.pvalues[var])

df_comparison = pd.DataFrame(comparison_data)

# Aggiungi le statistiche del modello
model_stats = {
    'Variable': ['Adj. R-squared', 'Num. Observations'],
    'Model_Parsimonious_Coeff': [results_parsimonious.rsquared_adj, results_parsimonious.nobs],
    'Model_Full_Coeff': [results_full.rsquared_adj, results_full.nobs],
    'Model_Parsimonious_PValue': ['', ''],
    'Model_Full_PValue': ['', ''],
}
df_comparison = pd.concat([df_comparison, pd.DataFrame(model_stats)], ignore_index=True)

print("\n" + "="*20 + " TABELLA DI CONFRONTO DEI MODELLI " + "="*20)
from tabulate import tabulate
print(tabulate(df_comparison, headers='keys', tablefmt='grid', floatfmt=".4f", showindex=False))
print("="*76)

01_yield_regressions.ipynb:cell3:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
01_yield_regressions.ipynb:cell3:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
Dati principali e dati BTC caricati con successo.
DataFrame uniti con successo.
Analisi condotta su 1715 osservazioni giornaliere.

Esecuzione del Modello Parsimonioso (solo risk-free)...

Esecuzione del Modello Completo (con tutti i controlli)...

========================= MODELLO 1: PARSIMONIOSO =========================
                            OLS Regression Results                            
Dep. Variable:                  3pool   R-squared:                       0.088
Model:                            OLS   Adj. R-squared:                  0.

## Curve VIF

Original calculation; depends on the preceding regression cell.

Source: `AAVE v2/REGRESSIONDAUSARE.ipynb`, original cell 6 (zero-based).

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Usa la matrice X che hai già creato per la regressione (senza la costante)
X_for_vif = X_full.drop('const', axis=1)

# Calcola il VIF per ogni variabile
vif_data = pd.DataFrame()
vif_data["feature"] = X_for_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_for_vif.values, i) for i in range(len(X_for_vif.columns))]

print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data)


--- Variance Inflation Factor (VIF) ---
          feature       VIF
0           3m ir  2.089849
1    ethperpetual  1.644428
2        gas_fees  3.045334
3             VIX  1.107288
4      btc_return  1.122109
5  volatility_btc  4.128333


## Lending: thesis specification

The saved source uses future 2y, while the submitted Table 3.6 uses 3m ir. Only that benchmark setting is restored. All coefficients, z statistics, adjusted R-squared, and N=1,715 reproduce the submitted table at its printed precision. This is a recovered configuration, not an untouched historical cell. The saved output in archived_outputs belongs to the future-2y variant.

Source: `AAVE v2/REGRESSIONDAUSARE.ipynb`, original cell 1 (zero-based).

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1. CONFIGURAZIONE E CARICAMENTO DATI ---

main_data_filename = str(_DATA_DIR / 'aavegas.csv')
btc_data_filename =  str(_DATA_DIR / 'btc_filtered.csv')
date_col = 'date'
dependent_var = 'stablecoin'

# Definiamo le variabili per i due modelli
vars_parsimonious = ['3m ir']    
vars_full = ['3m ir', 'ethperpetual', 'gas_fees', 'VIX', 'btc_return', 'volatility_btc']# ---------------------------------------------------

try:
    # Carica il DataFrame principale
    df_main = pd.read_csv(main_data_filename, parse_dates=[date_col], dayfirst=True)
    df_main.set_index(date_col, inplace=True)
    
    # Carica il DataFrame con i dati di Bitcoin
    df_btc = pd.read_csv(btc_data_filename, parse_dates=["Date"], dayfirst=True)
    df_btc.set_index("Date", inplace=True)
    
    print("Dati principali e dati BTC caricati con successo.")

    df = df_main.join(df_btc, how='inner')
    df.sort_index(inplace=True)
    
    print("DataFrame uniti con successo.")

except Exception as e:
    print(f"ERRORE durante il caricamento o la fusione dei dati: {e}")
    exit()

# Rimuovi righe con dati mancanti per il modello completo (per avere lo stesso campione)
df.dropna(subset=[dependent_var] + vars_full, inplace=True)
print(f"Analisi condotta su {len(df)} osservazioni giornaliere.")

# --- 2. PREPARAZIONE DELLE VARIABILI ---

# Variabile dipendente (Y)
y = df[dependent_var]

# Variabili indipendenti (X) per entrambi i modelli
X_parsimonious = sm.add_constant(df[vars_parsimonious])
X_full = sm.add_constant(df[vars_full])


# --- 3. ESECUZIONE DELLE REGRESSIONI ---

print("\nEsecuzione del Modello Parsimonioso (solo risk-free)...")
model_parsimonious = sm.OLS(y, X_parsimonious)
results_parsimonious = model_parsimonious.fit(cov_type='HAC', cov_kwds={'maxlags': 7})

print("\nEsecuzione del Modello Completo (con tutti i controlli)...")
model_full = sm.OLS(y, X_full)
results_full = model_full.fit(cov_type='HAC', cov_kwds={'maxlags': 7})


# --- 4. VISUALIZZAZIONE COMPARATIVA DEI RISULTATI ---

print("\n" + "="*25 + " MODELLO 1: PARSIMONIOSO " + "="*25)
print(results_parsimonious.summary())
print("="*75)

print("\n" + "="*25 + " MODELLO 2: COMPLETO " + "="*27)
print(results_full.summary())
print("="*75)

# --- CREAZIONE DI UNA TABELLA DI CONFRONTO ---

# Estrai i risultati chiave in un formato pulito
comparison_data = {
    'Variable': [],
    'Model_Parsimonious_Coeff': [],
    'Model_Parsimonious_PValue': [],
    'Model_Full_Coeff': [],
    'Model_Full_PValue': [],
}

# Aggiungi i coefficienti
for var in vars_parsimonious:
    comparison_data['Variable'].append(var)
    comparison_data['Model_Parsimonious_Coeff'].append(results_parsimonious.params[var])
    comparison_data['Model_Parsimonious_PValue'].append(results_parsimonious.pvalues[var])
    comparison_data['Model_Full_Coeff'].append(results_full.params.get(var, ''))
    comparison_data['Model_Full_PValue'].append(results_full.pvalues.get(var, ''))

for var in vars_full:
    if var not in vars_parsimonious:
        comparison_data['Variable'].append(var)
        comparison_data['Model_Parsimonious_Coeff'].append('')
        comparison_data['Model_Parsimonious_PValue'].append('')
        comparison_data['Model_Full_Coeff'].append(results_full.params[var])
        comparison_data['Model_Full_PValue'].append(results_full.pvalues[var])

df_comparison = pd.DataFrame(comparison_data)

# Aggiungi le statistiche del modello
model_stats = {
    'Variable': ['Adj. R-squared', 'Num. Observations'],
    'Model_Parsimonious_Coeff': [results_parsimonious.rsquared_adj, results_parsimonious.nobs],
    'Model_Full_Coeff': [results_full.rsquared_adj, results_full.nobs],
    'Model_Parsimonious_PValue': ['', ''],
    'Model_Full_PValue': ['', ''],
}
df_comparison = pd.concat([df_comparison, pd.DataFrame(model_stats)], ignore_index=True)

print("\n" + "="*20 + " TABELLA DI CONFRONTO DEI MODELLI " + "="*20)
from tabulate import tabulate
print(tabulate(df_comparison, headers='keys', tablefmt='grid', floatfmt=".4f", showindex=False))
print("="*76)

01_yield_regressions.ipynb:cell7:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
01_yield_regressions.ipynb:cell7:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
Dati principali e dati BTC caricati con successo.
DataFrame uniti con successo.
Analisi condotta su 1715 osservazioni giornaliere.

Esecuzione del Modello Parsimonioso (solo risk-free)...

Esecuzione del Modello Completo (con tutti i controlli)...

========================= MODELLO 1: PARSIMONIOSO =========================
                            OLS Regression Results                            
Dep. Variable:             stablecoin   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.

## Lending VIF

Original VIF calculation, run after the restored lending regression. The archived Table 5.4 uses future 2y (VIF 1.004); the current calculation correctly reports the restored benchmark. Both source/output versions are described in CODE_MAP.md.

Source: `AAVE v2/REGRESSIONDAUSARE.ipynb`, original cell 3 (zero-based).

In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Usa la matrice X che hai già creato per la regressione (senza la costante)
X_for_vif = X_full.drop('const', axis=1)

# Calcola il VIF per ogni variabile
vif_data = pd.DataFrame()
vif_data["feature"] = X_for_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_for_vif.values, i) for i in range(len(X_for_vif.columns))]

print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data)


--- Variance Inflation Factor (VIF) ---
          feature       VIF
0           3m ir  2.089849
1    ethperpetual  1.644428
2        gas_fees  3.045334
3             VIX  1.107288
4      btc_return  1.122109
5  volatility_btc  4.128333
